# Index coverage as a controlled variable in lexical vs dense code retrieval

Results notebook. It loads committed result files from the project repository and rebuilds every table and figure in the writeup. It runs no retrieval and needs no internet beyond cloning the repository.

Repository: https://github.com/pupa3066/context-configuration-builder

Design in one paragraph. A lexical retriever (BM25) searches every chunk of a corpus. A dense retriever indexes only a budget of chunks. When the budget is small, the two are compared on unequal ground. Here the dense index budget is varied directly while the corpus, questions and metrics stay fixed. BM25 is an internal control: its score cannot depend on the dense budget, so it must stay constant across budgets within a run.

Metrics: m1 is substring match of the target symbol in the retrieved chunks (favors lexical). m2 requires the symbol on a definition line (neither favored). m3 is m1 or a cosine credit above a threshold (favors dense).


In [ ]:
import json, os, glob, subprocess, platform
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REF = 'feat/agent-track-submission'
subprocess.run('rm -rf /tmp/cck && git clone --depth=1 --branch ' + REF +
               ' https://github.com/pupa3066/context-configuration-builder.git /tmp/cck', shell=True, check=True)
HEAD = subprocess.run('git -C /tmp/cck rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
BM = '/tmp/cck/benchmark'
M = ('m1_substring', 'm2_defline', 'm3_sem_credit')

# Only these single-run sweep files are read. Add a file name here only after confirming it was
# produced by the committed runner (conceptual_fair_metrics.py) on the full N=240 question set.
ALLOW_SWEEPS = ['SWEEP_cap_1000.json', 'SWEEP_cap_100000.json']
# Threshold sensitivity runs (full corpus, committed runner, N=240), reported in their own table.
ALLOW_EMBED = ['SWEEP_bge_base_full_N240.json']
ALLOW_THRESH = ['SWEEP_threshold_0.65_N240.json', 'SWEEP_threshold_0.80_N240.json']

print('Environment: python', platform.python_version(), '| pandas', pd.__version__, '| numpy', np.__version__)
print('Repository commit read by this notebook:', HEAD)


In [ ]:
def fmt(x):
    return f'{x[0]:.3f} [{x[1]:.3f}, {x[2]:.3f}]'
def overlap(a, b):
    return a[1] <= b[2] and b[1] <= a[2]


## Table 1. Cap sweep, committed runner, bge-small-en-v1.5, k=6, N=240

Each cell is the mean with a 95% bootstrap interval (1,000 resamples). `overlap` means the two intervals overlap; it shows no detected difference, not equivalence.


In [ ]:
rows = []
for f in ALLOW_SWEEPS:
    p = os.path.join(BM, f)
    if not os.path.exists(p):
        print('missing:', f); continue
    d = json.load(open(p))
    assert d['N'] == 240, f"{f}: N={d['N']}, expected 240"
    for m in M:
        b, s = d['arms']['bm25'][m], d['arms']['semantic'][m]
        rows.append(dict(cap=d['max_chunks'], model=d['embed_model'], metric=m,
                         bm25=fmt(b), semantic=fmt(s), diff=round(b[0] - s[0], 4), overlap=overlap(b, s)))
t1 = pd.DataFrame(rows).sort_values(['metric', 'cap'])
display(t1) if len(t1) else print('no sweep files found')
if len(t1):
    t1.to_csv('/kaggle/working/table1_cap_sweep.csv', index=False); print('saved table1_cap_sweep.csv')


# Internal control: BM25 must be identical across caps within this run.
if len(t1):
    for m in M:
        vals = t1[t1.metric == m]['bm25'].unique()
        print(m, 'BM25 constant across caps:', len(vals) == 1)


## Table 1b. m3 threshold sensitivity at full corpus (bge-small-en-v1.5, k=6, N=240)

The cosine threshold changes only m3; m1 and m2 are listed as a consistency check and should equal the full-corpus rows of Table 1. The 0.72 row is from `SWEEP_cap_100000.json`.


In [ ]:
rows = []
for f in ['SWEEP_cap_100000.json'] + ALLOW_THRESH:
    p = os.path.join(BM, f)
    if not os.path.exists(p): print('missing:', f); continue
    d = json.load(open(p)); assert d['N'] == 240 and d['max_chunks'] == 100000, f
    for m in M:
        b, s = d['arms']['bm25'][m], d['arms']['semantic'][m]
        rows.append(dict(threshold=d['sem_threshold'], metric=m, bm25=fmt(b), semantic=fmt(s),
                         diff=round(b[0] - s[0], 4), overlap=overlap(b, s)))
display(pd.DataFrame(rows).sort_values(['metric', 'threshold']))
_tab = pd.DataFrame(rows).sort_values(['metric', 'threshold'])
if len(_tab):
    _tab.to_csv('/kaggle/working/table1b_threshold.csv', index=False); print('saved table1b_threshold.csv')



## Table 1c. Embedder size at full corpus (k=6, N=240, threshold 0.72)

bge-base-en-v1.5 has about four times the parameters of bge-small-en-v1.5. Same runner, corpus and questions.


In [ ]:
rows = []
for f in ['SWEEP_cap_100000.json'] + ALLOW_EMBED:
    d = json.load(open(os.path.join(BM, f))); assert d['N'] == 240 and d['max_chunks'] == 100000, f
    for m in M:
        b, s = d['arms']['bm25'][m], d['arms']['semantic'][m]
        rows.append(dict(embedder=d['embed_model'].split('/')[-1], metric=m, bm25=fmt(b), semantic=fmt(s),
                         diff=round(b[0] - s[0], 4), overlap=overlap(b, s)))
display(pd.DataFrame(rows).sort_values(['metric', 'embedder']))
_tab = pd.DataFrame(rows).sort_values(['metric', 'embedder'])
if len(_tab):
    _tab.to_csv('/kaggle/working/table1c_embedder.csv', index=False); print('saved table1c_embedder.csv')



## Table 2. Replication against the earlier run

The earlier run used the same code and question miner on an earlier clone of the three repositories (psf/requests, pallets/click, psf/black). Repository contents may have changed between clones, so small shifts are expected and the two runs are reported separately. Earlier values are from `benchmark/CURRENT_STATE.md` in the repository.


In [ ]:
EARLIER = {
    1000:   {'bm25': (0.8667, 0.8208, 0.9583), 'semantic': (0.4417, 0.2792, 0.7125)},
    100000: {'bm25': (0.8667, 0.8208, 0.9583), 'semantic': (0.8667, 0.8125, 0.9333)},
}
rows = []
for f in ALLOW_SWEEPS:
    p = os.path.join(BM, f)
    if not os.path.exists(p): continue
    d = json.load(open(p)); cap = d['max_chunks']
    for arm in ('bm25', 'semantic'):
        for i, m in enumerate(M):
            new = d['arms'][arm][m][0]; old = EARLIER[cap][arm][i]
            rows.append(dict(cap=cap, arm=arm, metric=m, earlier=old, this_run=new, delta=round(new - old, 4)))
display(pd.DataFrame(rows)) if rows else print('no data')
_tab = pd.DataFrame(rows)
if len(_tab):
    _tab.to_csv('/kaggle/working/table2_replication.csv', index=False); print('saved table2_replication.csv')



## Table 3. Chunk size as a second controlled factor

`confound_sweep.py` imports the committed question miner, BM25, chunker, retriever and metrics, and changes only the chunker (8 lines is the committed default). It saves per-question hits, so the BM25 minus dense difference is tested pairwise: a paired bootstrap interval and an exact McNemar test on the discordant questions.


In [ ]:
rows = []
files = sorted(glob.glob(os.path.join(BM, 'CONFOUND_*.json')))
for p in files:
    d = json.load(open(p))
    if d['N'] != 240:
        print('skipping (N != 240):', os.path.basename(p)); continue
    frac = np.mean(list(d['indexed_fraction'].values()))
    ts = d.get('chunk_token_stats', {})
    for m in M:
        b, s = d['summary']['arms']['bm25'][m], d['summary']['arms']['semantic'][m]
        pr = d['summary']['paired'][m]
        rows.append(dict(repos='/'.join(d['n_chunks']), chunker=d['chunker'], dense_mode=d.get('sem_mode', 'whole'), embedder=d['embed_model'].split('/')[-1], cap=d['cap'], indexed_fraction=round(float(frac), 3),
                         chunks_over_limit=ts.get('frac_over_limit'), metric=m,
                         bm25=fmt(b), dense=fmt(s), paired_diff=pr['mean_diff_bm25_minus_sem'],
                         paired_ci=pr['ci95'], bm25_only=pr['bm25_only'], dense_only=pr['sem_only'],
                         mcnemar_p=pr['mcnemar_exact_p']))
t3 = pd.DataFrame(rows)
if len(t3):
    display(t3.sort_values(['metric', 'repos', 'chunker', 'dense_mode', 'cap']))
if len(t3):
    t3.to_csv('/kaggle/working/table3_confounds.csv', index=False); print('saved table3_confounds.csv')

else:
    print('no CONFOUND_*.json files in the repository yet')


In [ ]:
if len(t3):
    sub = t3[(t3.metric == 'm1_substring') & (t3.repos == 'requests/click/black') & (t3.embedder == 'bge-small-en-v1.5')].copy()
    order = {'lines8': 0, 'lines32': 1, 'words400': 2}
    sub['o'] = sub['chunker'].map(order) * 10 + (sub['dense_mode'] == 'maxwin') * 5 + (sub['cap'] > 1000)
    sub = sub.sort_values('o')
    def mean_of(s): return float(s.split(' ')[0])
    sub['b'] = sub['bm25'].map(mean_of); sub['s'] = sub['dense'].map(mean_of)
    sub['label'] = (sub['chunker'] + sub['dense_mode'].map(lambda m: ' windowed' if m == 'maxwin' else '')
                    + '\ncap ' + sub['cap'].astype(str) + '\n(' + (sub['indexed_fraction'] * 100).round(0).astype(int).astype(str) + '% indexed)')
    x = np.arange(len(sub))
    fig, ax = plt.subplots(figsize=(max(8, 1.5 * len(sub)), 4.6))
    ax.bar(x - 0.18, sub['b'], 0.36, label='BM25 (all chunks)', color='#475569')
    ax.bar(x + 0.18, sub['s'], 0.36, label='Dense (budgeted index)', color='#2563eb')
    ax.set_xticks(x); ax.set_xticklabels(sub['label'], fontsize=8)
    ax.set_ylabel('m1 substring hit rate'); ax.set_ylim(0, 1.18)
    ax.set_title('m1 by chunk size and dense index budget, bge-small, N=240, k=6')
    ax.legend(loc='upper center', ncol=2, frameon=False); ax.grid(axis='y', alpha=0.3); fig.tight_layout()
    fig.savefig('/kaggle/working/fig_m1_chunker_cap.png', dpi=140)
    print('saved /kaggle/working/fig_m1_chunker_cap.png; percent indexed is the mean over the three repositories')
else:
    print('figure skipped: no confound results yet')


## Table 4. Equivalence at full coverage (paired TOST)

A difference that is not significant is not evidence of no difference. Two one-sided tests (TOST) at margin d conclude equivalence when the paired 90% bootstrap interval of BM25 minus dense lies inside (-d, d). `smallest_margin` is the tightest margin at which equivalence holds. Pooled combines both repository sets (N=480). 10,000 paired resamples, seed 0.


In [ ]:
import random
def paired_ci(a, b, level, nb=10000, seed=0):
    d = [x - y for x, y in zip(a, b)]; n = len(d); rng = random.Random(seed)
    ms = sorted(sum(d[rng.randrange(n)] for _ in range(n)) / n for _ in range(nb))
    lo = ms[int((1 - level) / 2 * nb)]; hi = ms[int((1 + level) / 2 * nb) - 1]
    return sum(d) / n, lo, hi
sets = {
  'requests/click/black': 'CONFOUND_lines8_cap100000.json',
  'flask/httpx/rich': 'CONFOUND_lines8_newrepos_cap100000.json',
}
H = {k: json.load(open(os.path.join(BM, f)))['hits'] for k, f in sets.items()}
H['pooled (N=480)'] = {arm: {m: H['requests/click/black'][arm][m] + H['flask/httpx/rich'][arm][m] for m in M} for arm in ('bm25', 'semantic')}
rows = []
for k, h in H.items():
    for m in M:
        a, b = h['bm25'][m], h['semantic'][m]
        mean, lo90, hi90 = paired_ci(a, b, 0.90)
        _, lo95, hi95 = paired_ci(a, b, 0.95)
        sm = max(abs(lo90), abs(hi90))
        rows.append(dict(repos=k, metric=m, N=len(a), diff=round(mean, 4), ci90=[round(lo90, 4), round(hi90, 4)],
                         ci95=[round(lo95, 4), round(hi95, 4)], equivalent_at_0_10=sm < 0.10,
                         equivalent_at_0_05=sm < 0.05, smallest_margin=round(sm, 4)))
t4 = pd.DataFrame(rows); display(t4)
if len(t4):
    t4.to_csv('/kaggle/working/table4_equivalence.csv', index=False); print('saved table4_equivalence.csv')



## Table 5. Whole-chunk vs windowed dense scoring on identical questions (400-word chunks, full coverage)

Same chunks, same questions, same text returned to the metrics; only the dense scoring changes, from one truncated embedding per chunk to the best of several windows that fit the embedder. BM25 hits must be identical across the two files.


In [ ]:
import math
w = json.load(open(os.path.join(BM, 'CONFOUND_words400_cap100000.json')))
x = json.load(open(os.path.join(BM, 'CONFOUND_words400_maxwin_cap100000.json')))
assert w['hits']['bm25'] == x['hits']['bm25'], 'BM25 hits differ: question sets are not aligned'
print('BM25 hits identical across the two runs:', True)
rows = []
for m in M:
    a, b = w['hits']['semantic'][m], x['hits']['semantic'][m]
    up = sum(1 for i, j in zip(a, b) if j and not i); dn = sum(1 for i, j in zip(a, b) if i and not j); n = up + dn
    p = min(1.0, 2 * sum(math.comb(n, k) for k in range(min(up, dn) + 1)) / 2 ** n) if n else 1.0
    g, lo, hi = paired_ci(b, a, 0.95)
    gw = w['summary']['paired'][m]['mean_diff_bm25_minus_sem']; gx = x['summary']['paired'][m]['mean_diff_bm25_minus_sem']
    rows.append(dict(metric=m, dense_whole=round(sum(a) / len(a), 4), dense_windowed=round(sum(b) / len(b), 4),
                     gain=round(g, 4), gain_ci95=[round(lo, 4), round(hi, 4)], fixed=up, broken=dn, mcnemar_p=f'{p:.2e}',
                     bm25_gap_whole=gw, bm25_gap_windowed=gx, share_of_gap_removed=round((gw - gx) / gw, 3)))
display(pd.DataFrame(rows))
_tab = pd.DataFrame(rows)
if len(_tab):
    _tab.to_csv('/kaggle/working/table5_windowed.csv', index=False); print('saved table5_windowed.csv')

for f in ('CONFOUND_words400_cap100000.json', 'CONFOUND_words400_maxwin_cap100000.json'):
    pr = json.load(open(os.path.join(BM, f))).get('window_probe')
    if pr: print('embedder window probe (bge-small):', pr)


## Table 6. Gold-chunk coverage decomposition

For each question, `gold indexed` records whether a chunk holding the answer's definition line was in the dense index at that cap. If coverage explains the gap, dense accuracy among questions whose gold chunk was indexed should stay roughly the same at every cap, while the overall rate falls with the fraction indexed.


In [ ]:
rows = []
for p in sorted(glob.glob(os.path.join(BM, 'CONFOUND_lines8_gold*_cap*.json'))):
    d = json.load(open(p)); g = d.get('gold')
    if not g or d['N'] != 240: continue
    for m in ('m1_substring', 'm2_defline'):
        gi, gm = g['dense_given_gold_indexed'][m], g['dense_given_gold_missing'][m]
        rows.append(dict(repos='/'.join(d['n_chunks']), cap=d['cap'], metric=m, frac_gold_indexed=g['frac_gold_indexed'],
                         dense_overall=d['summary']['arms']['semantic'][m][0],
                         dense_given_gold_indexed=gi['rate'], n_indexed=gi['n'],
                         dense_given_gold_missing=gm['rate'], n_missing=gm['n']))
t6 = pd.DataFrame(rows)
display(t6.sort_values(['repos', 'metric', 'cap'])) if len(t6) else print('no gold-chunk files in the repository yet')
if len(t6):
    t6.to_csv('/kaggle/working/table6_gold.csv', index=False); print('saved table6_gold.csv')



## Paper figures


In [ ]:
# Paper figures, rebuilt from committed result files only.
import json, os, glob, math, random
import numpy as np, matplotlib.pyplot as plt
OUTD = '/kaggle/working'
def J(f): return json.load(open(os.path.join(BM, f)))
BLUE, GREY, ORANGE = '#2563eb', '#475569', '#ea580c'
def m(x): return x[0]
def err(x): return [[x[0]-x[1]], [x[2]-x[0]]]

# Figure 1: coverage. Dense m1 overall vs given the gold chunk was indexed, BM25 as the control.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (tag, title) in zip(axes, [('gold', 'requests, click, black'), ('gold-newrepos', 'flask, httpx, rich')]):
    rows = []
    for cap in (1000, 3000, 100000):
        d = J(f'CONFOUND_lines8_{tag}_cap{cap}.json'); g = d['gold']; s = d['summary']['arms']
        rows.append((g['frac_gold_indexed'], s['semantic']['m1_substring'], g['dense_given_gold_indexed']['m1_substring']['rate'], s['bm25']['m1_substring']))
    x = [r[0] * 100 for r in rows]
    ax.errorbar(x, [m(r[1]) for r in rows], yerr=np.hstack([err(r[1]) for r in rows]), color=BLUE, marker='o', lw=2, capsize=3, label='Dense, all questions')
    ax.plot(x, [r[2] for r in rows], color=BLUE, marker='s', ls='--', lw=1.5, label='Dense, gold chunk indexed')
    ax.errorbar(x, [m(r[3]) for r in rows], yerr=np.hstack([err(r[3]) for r in rows]), color=GREY, marker='o', lw=2, capsize=3, label='BM25 (always all chunks)')
    for xi, r in zip(x, rows): ax.annotate(f'{xi:.0f}%', (xi, m(r[1])), textcoords='offset points', xytext=(0, -14), ha='center', fontsize=8)
    ax.set_title(title); ax.set_xlabel('Questions whose gold chunk is in the dense index (%)'); ax.set_xlim(25, 105); ax.set_ylim(0.3, 1.02); ax.grid(alpha=0.3)
axes[0].set_ylabel('m1 hit rate (95% CI)'); axes[0].legend(loc='lower right', fontsize=8)
fig.suptitle('Figure 1. Dense accuracy follows index coverage; with the answer indexed it is flat (N=240 per set)', fontsize=10)
fig.tight_layout(); fig.savefig(f'{OUTD}/fig1_coverage.png', dpi=150); print('saved fig1_coverage.png')

# Figure 2: equivalence at full coverage, paired BM25 minus dense with 90% and 95% intervals.
def pci(a, b, level, nb=10000, seed=0):
    d = [x - y for x, y in zip(a, b)]; n = len(d); rng = random.Random(seed)
    ms = sorted(sum(d[rng.randrange(n)] for _ in range(n)) / n for _ in range(nb))
    return sum(d) / n, ms[int((1 - level) / 2 * nb)], ms[int((1 + level) / 2 * nb) - 1]
H1 = J('CONFOUND_lines8_cap100000.json')['hits']; H2 = J('CONFOUND_lines8_newrepos_cap100000.json')['hits']
HP = {a: {k: H1[a][k] + H2[a][k] for k in M} for a in ('bm25', 'semantic')}
sets = [('Set 1', H1), ('Set 2', H2), ('Pooled', HP)]
fig, ax = plt.subplots(figsize=(8, 4.2)); yl = []; y = 0
for k in M:
    for name, h in sets:
        mu, l90, h90 = pci(h['bm25'][k], h['semantic'][k], 0.90); _, l95, h95 = pci(h['bm25'][k], h['semantic'][k], 0.95)
        ax.plot([l95, h95], [y, y], color=GREY, lw=1); ax.plot([l90, h90], [y, y], color=BLUE, lw=4); ax.plot(mu, y, 'o', color='black', ms=4)
        yl.append(f'{k.split("_")[0]}  {name}'); y += 1
    y += 0.6
ax.axvline(0, color='black', lw=0.8); ax.axvspan(-0.10, 0.10, color='#16a34a', alpha=0.08)
ax.axvline(-0.10, color='#16a34a', ls='--', lw=1); ax.axvline(0.10, color='#16a34a', ls='--', lw=1)
ax.set_yticks([i + (i // 3) * 0.6 for i in range(len(yl))]); ax.set_yticklabels(yl, fontsize=8); ax.invert_yaxis()
ax.set_xlabel('BM25 minus dense at full coverage (thick: 90% CI, thin: 95% CI)')
ax.set_title('Figure 2. Equivalent within the 0.10 margin on every metric (TOST), 8-line chunks', fontsize=10)
fig.tight_layout(); fig.savefig(f'{OUTD}/fig2_equivalence.png', dpi=150); print('saved fig2_equivalence.png')

# Figure 3: chunk length at full coverage, plus the windowed rescue on identical chunks.
conds = [('8 lines', 'CONFOUND_lines8_cap100000.json'), ('32 lines', 'CONFOUND_lines32_cap100000.json'),
         ('400 words', 'CONFOUND_words400_cap100000.json'), ('400 words,\nwindowed dense', 'CONFOUND_words400_maxwin_cap100000.json')]
fig, ax = plt.subplots(figsize=(9, 4.3)); x = np.arange(len(conds))
for i, (lab, f) in enumerate(conds):
    d = J(f); s = d['summary']['arms']; b, dn = s['bm25']['m1_substring'], s['semantic']['m1_substring']
    ax.bar(i - 0.19, m(b), 0.36, yerr=err(b), capsize=3, color=GREY, label='BM25' if i == 0 else None)
    ax.bar(i + 0.19, m(dn), 0.36, yerr=err(dn), capsize=3, color=BLUE, label='Dense' if i == 0 else None)
    over = d['chunk_token_stats'].get('frac_over_limit')
    ax.text(i, 0.06, f"{over*100:.1f}% of chunks\nover 512 tokens" if over is not None else '', ha='center', fontsize=7.5, color='white')
ax.set_xticks(x); ax.set_xticklabels([c[0] for c in conds], fontsize=9); ax.set_ylim(0, 1.12); ax.set_ylabel('m1 hit rate (95% CI)')
ax.legend(loc='upper left', ncol=2, frameon=False); ax.grid(axis='y', alpha=0.3)
ax.set_title('Figure 3. At full coverage, dense accuracy falls with chunk length; windowing recovers most of it', fontsize=10)
fig.tight_layout(); fig.savefig(f'{OUTD}/fig3_chunk_length.png', dpi=150); print('saved fig3_chunk_length.png')

# Figure 4: embedder window probe (bge-small) and the paired windowed rescue.
w = J('CONFOUND_words400_cap100000.json'); xw = J('CONFOUND_words400_maxwin_cap100000.json')
pr = w['window_probe']['cosine_by_filler_words']
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
xs = sorted(int(k) for k in pr); a1.plot(xs, [pr[str(k)] for k in xs], 'o-', color=ORANGE, lw=2)
a1.axhline(1.0, color='black', lw=0.6, ls=':'); a1.set_xscale('log'); a1.set_ylim(0.7, 1.02)
a1.set_xlabel('Filler words before the differing tail (log scale)'); a1.set_ylabel('Cosine between the two embeddings')
a1.set_title('Tail ignored after about 512 tokens (cosine = 1.000)', fontsize=9); a1.grid(alpha=0.3)
fx, br = [], []
for k in M:
    A, B = w['hits']['semantic'][k], xw['hits']['semantic'][k]
    fx.append(sum(1 for i, j in zip(A, B) if j and not i)); br.append(sum(1 for i, j in zip(A, B) if i and not j))
xx = np.arange(len(M)); a2.bar(xx - 0.19, fx, 0.36, color='#16a34a', label='fixed by windowing'); a2.bar(xx + 0.19, br, 0.36, color='#dc2626', label='broken by windowing')
for i in range(len(M)): a2.text(xx[i] - 0.19, fx[i] + 1, str(fx[i]), ha='center', fontsize=8); a2.text(xx[i] + 0.19, br[i] + 1, str(br[i]), ha='center', fontsize=8)
a2.set_xticks(xx); a2.set_xticklabels([k.split('_')[0] for k in M]); a2.set_ylabel('Questions (of 240)'); a2.legend(frameon=False)
a2.set_title('Same chunks and questions, only dense scoring changed', fontsize=9); a2.grid(axis='y', alpha=0.3)
fig.suptitle('Figure 4. The embedder window, measured, and the paired effect of removing truncation', fontsize=10)
fig.tight_layout(); fig.savefig(f'{OUTD}/fig4_window.png', dpi=150); print('saved fig4_window.png')

# Card image, 560 x 280.
fig, ax = plt.subplots(figsize=(5.6, 2.8), dpi=100)
rows = [J(f'CONFOUND_lines8_gold_cap{c}.json') for c in (1000, 3000, 100000)]
xg = [r['gold']['frac_gold_indexed'] * 100 for r in rows]
ax.plot(xg, [r['summary']['arms']['semantic']['m1_substring'][0] for r in rows], 'o-', color=BLUE, lw=2.5, label='Dense')
ax.plot(xg, [r['summary']['arms']['bm25']['m1_substring'][0] for r in rows], 'o-', color=GREY, lw=2.5, label='BM25')
ax.set_xlabel('Answer chunks in the dense index (%)', fontsize=8); ax.set_ylabel('Hit rate', fontsize=8); ax.tick_params(labelsize=7)
ax.set_title('Index coverage, not the retriever, made the gap', fontsize=9); ax.legend(fontsize=7, frameon=False); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(f'{OUTD}/card_560x280.png', dpi=100); print('saved card_560x280.png')


In [ ]:
# Validity checks printed for the reader.
print('1. N=240 asserted for every file used.')
if len(t3):
    for key, g in t3[t3.metric == 'm1_substring'].groupby(['repos', 'chunker']):
        print('2. BM25 m1 constant across caps for', key, ':', g['bm25'].nunique() == 1)
print('3. All files come from repository commit', HEAD)


## Scope

- 240 questions mined from docstrings of three Python repositories. Docstring-mined questions share vocabulary with the target text, which favors lexical matching. Human-written queries, other languages and code-specialized embedders were not tested here.
- m1 and m3 reward retrieving the target symbol, not solving a task. These results do not measure agent task success.
- Overlapping intervals show no detected difference, not equivalence. Table 3 adds paired tests for that reason.
- Only the files listed in the setup cell are read. A result is included only if it came from the committed code.
